# Experiment 03: Tuning lgbm_building

**Question:** how much does tuning improve lgbm_building over its untuned settings (500 trees, learning rate 0.1,
63 leaves; 1.217 in [exp01_building.ipynb](exp01_building.ipynb))?

Card: [exp03_tuning.md](../experiments/exp03_tuning.md).

- **Fixed:** lgbm_building (33 shared features + `building_id` as a categorical), learning rate 0.1, 2M training /
  500K validation rows per fold ([exp02](exp02_training_rows.ipynb): more rows don't help), the 4 seasonal folds, seed 0.
- **Every trial** trains on all 4 folds with up to 5,000 trees. Each fold stops once validation RMSLE hasn't improved
  for 200 trees and records its best tree count; the trial's tree count is the mean over the folds. The full validation
  curves go to MLflow.
- **Stage A, tree size:** 31, 63, 127, 255 leaves.
- **Stage B, regularization:** on top of the best Stage A trial, one change at a time: `min_child_samples` 100 and 500,
  `colsample_bytree` 0.8, smoother `building_id` categories (`cat_smooth` 50, `min_data_per_group` 500), and finer ones
  (`max_cat_threshold` 128).
- **Final:** the best trial retrained with its tree count, scored like exp01 (all rows, clean rows, per meter).

A first attempt with learning rate 0.05 and a fixed 1,500 trees was stopped: every curve was still falling at 1,500
trees, so the comparison favoured the faster learning rate. Its runs were deleted.

Tree counts and settings are chosen on the same folds they are scored on, so the tuned score is slightly optimistic.
Every trial is logged to MLflow, experiment `ashrae-baseline-tuning`. Each fit shows a progress bar with the current
and best validation RMSLE.


In [ ]:
import sys
from time import perf_counter, strftime

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.entities import Metric
from tqdm import tqdm

sys.path.insert(0, "../..")  # the ashrae package lives in the homework folder
from ashrae.config import FOLDS, METERS, MLFLOW_URI
from ashrae.data import read_processed, split
from ashrae.features import build_features
from ashrae.metrics import rmsle, rmsle_by_meter
from ashrae.tracking import log_run

EXPERIMENT = "ashrae-baseline-tuning"
SESSION = strftime("%Y-%m-%d %H:%M:%S")  # tag shared by this execution's runs
N_TRAIN, N_VAL = (
    2_000_000,
    500_000,
)  # rows sampled per fold, as in exp01 (exp02: more rows don't help)
LEARNING_RATE = 0.1
MAX_TREES = 5000
PATIENCE = 200  # trees without improvement before a fold stops
CURVE_EVERY = 10  # every 10th tree of each validation curve goes to MLflow

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
client = mlflow.MlflowClient()

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/11 12:25:43 INFO mlflow.tracking.fluent: Experiment with name 'ashrae-baseline-tuning' does not exist. Creating a new experiment.


## Fold data

Built once and shared by every trial.


In [ ]:
df = read_processed()
BUILDING = pd.CategoricalDtype(
    range(int(df.building_id.max()) + 1)
)  # all ids, so every fold shares categories


def with_building(X, rows):
    """`X` plus `building_id` as a categorical, which LightGBM splits on directly."""
    return X.assign(
        building_id=rows.building_id.astype(int).astype(BUILDING).to_numpy()
    )


folds = {}
for fold, months in tqdm(FOLDS.items(), desc="fold data", unit="fold"):
    train_df, val_df = split(df, months, N_TRAIN, N_VAL)
    year_median = train_df.year_built.median()
    folds[fold] = {
        "X_train": with_building(build_features(train_df, year_median), train_df),
        "y_train": np.log1p(train_df.meter_reading.to_numpy(dtype="float32")),
        "X_val": with_building(build_features(val_df, year_median), val_df),
        "y_val": np.log1p(val_df.meter_reading.to_numpy(dtype="float32")),
        "clean": ~val_df.anomaly.to_numpy(),
        "meter": val_df.meter.to_numpy(),
    }
del df, train_df, val_df

fold data: 100%|██████████| 4/4 [00:05<00:00,  1.25s/fold]


## Trials

`progress` draws one bar per fit, with the current and best validation RMSLE, and stops the fit after `PATIENCE`
trees without improvement on all validation rows (LightGBM's own early stopping would also stop on the clean-rows
set). `run_trial` trains one setting on every fold and logs one MLflow run per fold: the setting, the fold's best tree
count, validation RMSLE there (all rows and clean rows), training time, and both validation curves. The curves use
LightGBM's `rmse` on `log1p` values, which equals RMSLE except that negative predictions aren't clipped at 0.


In [ ]:
def progress(desc, total, patience=None):
    """LightGBM callback: a progress bar with validation RMSLE, and early stopping on the "all" validation set."""
    bar = tqdm(total=total, desc=desc, unit="tree", dynamic_ncols=True)
    best = {"score": np.inf, "tree": 0}

    def callback(env):
        trees = env.iteration + 1
        score = {name: value for name, _, value, _ in env.evaluation_result_list}["all"]
        if score < best["score"]:
            best.update(score=score, tree=trees)
        bar.update(1)
        if trees % 10 == 0:
            bar.set_postfix_str(
                f"val {score:.4f}, best {best['score']:.4f} at {best['tree']}"
            )
        stop = patience is not None and trees - best["tree"] >= patience
        if stop or trees == total:
            bar.set_postfix_str(
                f"best {best['score']:.4f} at {best['tree']}"
                + (" (early stop)" if stop else "")
            )
            bar.total = trees
            bar.close()
        if stop:
            raise lgb.callback.EarlyStopException(
                best["tree"] - 1, env.evaluation_result_list
            )

    callback.order = 30  # after record_evaluation, so the curve includes the last tree
    return callback


def run_trial(name, stage, params):
    """Trains `params` on every fold; returns the mean of the folds' best tree counts and their mean RMSLE."""
    curves, seconds = {}, {}
    for fold, d in folds.items():
        history = {}
        start = perf_counter()
        lgb.LGBMRegressor(
            n_estimators=MAX_TREES,
            learning_rate=LEARNING_RATE,
            metric="rmse",
            random_state=0,
            verbose=-1,
            **params,
        ).fit(
            d["X_train"],
            d["y_train"],
            eval_X=(d["X_val"], d["X_val"][d["clean"]]),
            eval_y=(d["y_val"], d["y_val"][d["clean"]]),
            eval_names=["all", "clean"],
            callbacks=[
                lgb.record_evaluation(history),
                progress(f"{name} | {fold}", MAX_TREES, PATIENCE),
            ],
        )
        seconds[fold] = perf_counter() - start
        curves[fold] = {
            rows: np.array(history[rows]["rmse"]) for rows in ["all", "clean"]
        }
    best = {
        fold: int(c["all"].argmin()) for fold, c in curves.items()
    }  # index of each fold's best tree
    trees = round(np.mean([b + 1 for b in best.values()]))
    for fold, c in curves.items():
        with mlflow.start_run(
            run_name=f"{name} | {fold}",
            tags={
                "session": SESSION,
                "fold": fold,
                "trial": name,
                "stage": stage,
                "model": "lgbm_building",
                "notebook": "exp03_tuning",
            },
        ) as run:
            mlflow.log_params(
                {
                    "learning_rate": LEARNING_RATE,
                    **params,
                    "max_trees": MAX_TREES,
                    "patience": PATIENCE,
                    "n_train": N_TRAIN,
                    "n_val": N_VAL,
                }
            )
            mlflow.log_metrics(
                {
                    "best_trees": best[fold] + 1,
                    "trial_trees": trees,
                    "val_rmsle": c["all"][best[fold]],
                    "val_rmsle_clean": c["clean"][best[fold]],
                    "train_seconds": seconds[fold],
                }
            )
            steps = range(CURVE_EVERY - 1, len(c["all"]), CURVE_EVERY)
            client.log_batch(
                run.info.run_id,
                metrics=[
                    Metric(f"val_rmsle_curve_{rows}", float(c[rows][i]), 0, i + 1)
                    for rows in ["all", "clean"]
                    for i in steps
                ],
            )
    score = float(np.mean([c["all"][best[fold]] for fold, c in curves.items()]))
    print(f"{name}: {trees} trees (fold mean), fold-average RMSLE {score:.4f}\n")
    return trees, score


trials = {}  # name -> (stage, params, trees, fold-average RMSLE)

### Stage A: tree size


In [ ]:
STAGE_A = {f"{leaves} leaves": {"num_leaves": leaves} for leaves in [31, 63, 127, 255]}
for name, params in STAGE_A.items():
    trials[name] = ("A", params, *run_trial(name, "A", params))

best_a = min((n for n in trials if trials[n][0] == "A"), key=lambda n: trials[n][3])
print(f"Stage A best: {best_a}")

31 leaves | Jan–Mar: 100%|██████████| 4541/4541 [00:57<00:00, 78.84tree/s, best 1.4077 at 4341 (early stop)]
31 leaves | Apr–Jun: 100%|██████████| 3950/3950 [00:49<00:00, 79.53tree/s, best 1.2042 at 3750 (early stop)]
31 leaves | Jul–Sep: 100%|██████████| 4802/4802 [01:02<00:00, 77.38tree/s, best 1.0319 at 4602 (early stop)]
31 leaves | Oct–Dec: 100%|██████████| 3004/3004 [00:41<00:00, 72.57tree/s, best 1.0947 at 2804 (early stop)]


31 leaves: 3874 trees (fold mean), fold-average RMSLE 1.1846



63 leaves | Jan–Mar: 100%|██████████| 3116/3116 [01:04<00:00, 48.16tree/s, best 1.4065 at 2916 (early stop)]
63 leaves | Apr–Jun: 100%|██████████| 2610/2610 [00:57<00:00, 45.14tree/s, best 1.2031 at 2410 (early stop)]
63 leaves | Jul–Sep:  10%|▉         | 482/5000 [00:13<02:05, 35.94tree/s, val 1.0923, best 1.0923 at 480]

### Stage B: regularization, on top of the best Stage A trial


In [ ]:
base = trials[best_a][1]
STAGE_B = {
    "min_child_samples 100": {"min_child_samples": 100},
    "min_child_samples 500": {"min_child_samples": 500},
    "colsample_bytree 0.8": {"colsample_bytree": 0.8},
    "smoother building categories": {"cat_smooth": 50, "min_data_per_group": 500},
    "finer building categories": {"max_cat_threshold": 128},
}
for name, change in STAGE_B.items():
    trials[f"{best_a} + {name}"] = (
        "B",
        {**base, **change},
        *run_trial(f"{best_a} + {name}", "B", {**base, **change}),
    )

best = min(trials, key=lambda n: trials[n][3])
print(f"Best trial: {best}, {trials[best][2]} trees")

## Final model

The best trial retrained with its tree count (no early stopping), scored and logged like exp01
(`lgbm_building_tuned | <fold>`).


In [ ]:
_, best_params, best_trees, _ = trials[best]
tuned = {"learning_rate": LEARNING_RATE, **best_params, "n_estimators": best_trees}
for fold, d in folds.items():
    start = perf_counter()
    model = lgb.LGBMRegressor(**tuned, metric="rmse", random_state=0, verbose=-1).fit(
        d["X_train"],
        d["y_train"],
        eval_X=(d["X_val"],),
        eval_y=(d["y_val"],),
        eval_names=["all"],
        callbacks=[progress(f"lgbm_building_tuned | {fold}", best_trees)],
    )
    train_seconds = perf_counter() - start
    pred = np.clip(model.predict(d["X_val"]), 0, None)
    metrics = {
        **rmsle_by_meter(d["y_val"], pred, d["meter"]),
        "val_rmsle_clean": rmsle(d["y_val"][d["clean"]], pred[d["clean"]]),
        "train_rmsle": rmsle(d["y_train"], model.predict(d["X_train"])),
        "train_seconds": train_seconds,
    }
    log_run(
        EXPERIMENT,
        f"lgbm_building_tuned | {fold}",
        params={**tuned, "n_train": N_TRAIN, "n_val": N_VAL, "from_trial": best},
        metrics=metrics,
        tags={
            "session": SESSION,
            "fold": fold,
            "model": "lgbm_building_tuned",
            "notebook": "exp03_tuning",
        },
    )
    print(
        f"{fold} lgbm_building_tuned val RMSLE {metrics['val_rmsle']:.3f} "
        f"(clean rows {metrics['val_rmsle_clean']:.3f}), {train_seconds:.0f} s"
    )

## Results

Read from MLflow, the latest session of this notebook, so this section runs without retraining. The untuned
lgbm_building comes from exp01 (`ashrae-baseline-building`). The figure is saved to
[figures/](../figures/exp03_tuning.png) and logged as an artifact of each run.


In [ ]:
def latest_session(experiment, notebook):
    runs = mlflow.search_runs(
        experiment_names=[experiment], filter_string=f"tags.notebook = '{notebook}'"
    )
    runs = runs[runs["tags.session"] == runs["tags.session"].max()]
    return runs.rename(
        columns=lambda c: c.removeprefix("metrics.")
        .removeprefix("tags.")
        .removeprefix("params.")
    )


runs = latest_session(EXPERIMENT, "exp03_tuning")
print(f"MLflow session {runs.session.iloc[0]}: {len(runs)} runs")
trial_runs = runs[runs.trial.notna()]
TRIAL_ORDER = list(dict.fromkeys(trial_runs.sort_values("start_time").trial))

log = (
    trial_runs.groupby("trial")
    .agg(
        stage=("stage", "first"),
        trees=("trial_trees", "first"),
        rmsle=("val_rmsle", "mean"),
        rmsle_clean=("val_rmsle_clean", "mean"),
        seconds=("train_seconds", "mean"),
    )
    .loc[TRIAL_ORDER]
)
log["trees"] = log.trees.astype(int)
best_a = log[log.stage == "A"].rmsle.idxmin()  # the trial Stage B builds on
display(log.round(4).rename_axis("tuning log, fold averages"))

In [ ]:
untuned = latest_session("ashrae-baseline-building", "exp01_building")
untuned = untuned[untuned.model == "lgbm_building"].assign(
    model="lgbm_building (untuned)"
)
final = pd.concat([untuned, runs[runs.model == "lgbm_building_tuned"]])
MODELS = ["lgbm_building (untuned)", "lgbm_building_tuned"]

table = final.pivot(index="model", columns="fold", values="val_rmsle").loc[
    MODELS, list(FOLDS)
]
averages = final.groupby("model")[
    ["val_rmsle", "val_rmsle_clean", "train_rmsle", "train_seconds"]
].mean()
table["average"] = averages.val_rmsle
table["average, clean rows"] = averages.val_rmsle_clean
table["training"] = averages.train_rmsle
table["seconds per fit"] = averages.train_seconds
display(table.round(3).rename_axis("RMSLE, all rows unless noted"))

meters = (
    final.groupby("model")[[f"val_rmsle_{name}" for name in METERS.values()]]
    .mean()
    .loc[MODELS]
)
display(
    meters.rename(columns=lambda c: c.removeprefix("val_rmsle_"))
    .round(3)
    .rename_axis("fold average, all rows")
)

In [ ]:
def average_curve(trial):
    """Fold-average validation curve of a trial from MLflow, up to the fold that stopped first."""
    histories = [
        sorted(
            client.get_metric_history(run_id, "val_rmsle_curve_all"),
            key=lambda m: m.step,
        )
        for run_id in trial_runs[trial_runs.trial == trial].run_id
    ]
    length = min(len(h) for h in histories)
    return [m.step for m in histories[0][:length]], np.mean(
        [[m.value for m in h[:length]] for h in histories], axis=0
    )


PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
untuned_score = averages.val_rmsle["lgbm_building (untuned)"]
ylim = (
    log.rmsle.min() - 0.01,
    max(untuned_score, log.rmsle.max()) + 0.02,
)  # around the chosen points
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, stage, title in zip(
    axes, ["A", "B"], ["Stage A: tree size", "Stage B: regularization"]
):
    names = [t for t in TRIAL_ORDER if log.stage[t] == stage]
    if stage == "B":  # the Stage A trial that Stage B builds on, for reference
        steps, values = average_curve(best_a)
        ax.plot(steps, values, color="black", lw=2, label=f"{best_a} (Stage A best)")
    for name, color in zip(names, PALETTE):
        steps, values = average_curve(name)
        ax.plot(
            steps, values, color=color, lw=2, label=name.removeprefix(f"{best_a} + ")
        )
    ax.axhline(
        untuned_score,
        color="gray",
        ls="--",
        lw=1,
        label=f"untuned, 500 trees ({untuned_score:.3f})",
    )
    ax.set(title=title, xlabel="trees", ylim=ylim)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("validation RMSLE, fold average")
plt.tight_layout()
fig.savefig("../figures/exp03_tuning.png", dpi=150, bbox_inches="tight")

# The figure covers the whole experiment, so every run of this session gets a copy
for run_id in runs.run_id:
    client.log_artifact(run_id, "../figures/exp03_tuning.png")

## Findings

_Filled in after the run._
